# Audit a LeRobot-format robot dataset

Runs `lerobot_audit` on a local LeRobot v3 folder (meta/ + data/ parquet, no videos). Download first with `python -I scripts/download.py <name>`. Change `DATASET` to `fineart` or `droid`.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from lerobot_audit.profiles import get_profile
from lerobot_audit.audit import run_audit
from lerobot_audit.acceptance import run_checks, load_spec
from lerobot_audit.report import write_report
DATASET = 'libero_10'

## 1. Audit
Episode lengths, idle time, smoothness, gripper events and re-grasps, duplicates, label hygiene, coverage, usable hours.

In [ ]:
p = get_profile(DATASET)
res = run_audit(p)
s = res.summary
{k: s[k] for k in ['slice', 'idle', 'smoothness', 'gripper', 'duplicates', 'usable']}

## 2. Per-episode table (worst idle first)

In [ ]:
res.episodes.sort_values('idle_frac', ascending=False)[['episode_index', 'task', 'duration_s', 'idle_frac', 'sparc', 'n_close', 'n_regrasp', 'nn_dist']].head(10)

## 3. Label hygiene and coverage

In [ ]:
{k: v for k, v in s['hygiene'].items() if not isinstance(v, (list, dict)) or len(str(v)) < 400}, s['coverage']

## 4. Acceptance gate and one-page report

In [ ]:
import json
info = json.loads((p.root / 'meta' / 'info.json').read_text())
checks = run_checks(res, load_spec(DATASET), info)
{n: (c.passed, len(c.flagged)) for n, c in checks.items()}

In [ ]:
out = ROOT / 'results' / 'notebook' / DATASET
write_report(res, p.root, out, checks)
from IPython.display import Image, Markdown
display(Image(filename=str(out / 'report.png')))
Markdown((out / 'report.md').read_text())